# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² Croissant dataset using the `mlcroissant` library, referencing dataset elements by their `@id` fields throughout.

### Dataset Source
The dataset metadata and schema are available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and initialize the `mlcroissant` Dataset object for further use.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not as dict or list)
meta = dataset.metadata
print(f"Dataset title: {meta.name}\n\nDescription: {meta.description}\n")

## 2. Data Overview

Let's inspect the available record sets and their fields. All outputs below will use the `@id` fields for referencing dataset components.

In [ ]:
from pprint import pprint

# List all record sets with their @id and names
_record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in _record_sets:
    print(f"  - @id: {rs['@id']}  name: {rs.get('name', '[no name]')}")

# For demonstration, display fields for the first record set (if available).
if _record_sets:
    first_record_set_id = _record_sets[0]['@id']
    print(f"\nFields in record set '@id': {first_record_set_id}")
    for field in _record_sets[0].get('fields', []):
        field_id = field.get('@id', '[no id]')
        name = field.get('name', '[no name]')
        dtype = field.get('dataType', '[no dataType]')
        print(f"    - Field @id: {field_id}, name: {name}, type: {dtype}")
else:
    print("No record sets found in the schema.")

## 3. Data Extraction

Load records from one or more record sets into pandas DataFrames. All references below use `@id` fields.

In [ ]:
# Identify all available record set @ids
record_set_ids = [_rs['@id'] for _rs in _record_sets]

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set: {rs_id}")

# Show columns from the first record set's DataFrame if any loaded
if record_set_ids and dataframes[record_set_ids[0]].shape[0] > 0:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("\nNo data available in record sets.")

## 4. Exploratory Data Analysis (EDA)

Select relevant fields and perform filtering, normalization, and grouping for analysis.

**All fields referenced below use their `@id` values.**

In [ ]:
# EDA on the main record set if available
import numpy as np

# Use first record set, replace these IDs with actual @ids as available in the dataset
if record_set_ids and dataframes[record_set_ids[0]].shape[0] > 0:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    
    # Find a numeric field by @id heuristically (user should inspect output and pick real one!)
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in (np.float64, np.int64) and not col.lower().startswith('unnamed')]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]  # by @id
    else:
        print('No numeric field found; update this cell with the appropriate field @id!')
        numeric_field_id = None
    
    if numeric_field_id:
        # Thresholding: pick an arbitrary threshold for demonstration
        threshold = df[numeric_field_id].quantile(0.9) if df[numeric_field_id].nunique() > 1 else df[numeric_field_id].min()
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalization (standard score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized field '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field if present
        # Try to use another column that might be categorical (not numeric)
        group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped (mean) by {group_field_id} for filtered records:")
            display(grouped.head())
        else:
            print('No suitable group field found; update as needed.')
    else:
        print("You can manually set 'numeric_field_id' = <@id of numeric field> to proceed.")
else:
    print("No record set data to analyze.")

## 5. Visualization

Visualize the distribution of a selected numeric field and relationships with a selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize if we found a numeric field
if record_set_ids and dataframes[record_set_ids[0]].shape[0] > 0 and 'numeric_field_id' in locals() and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field is available, boxplot by category
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric field available for visualization.")

## 6. Conclusion

In this notebook, we've:
- Loaded FAIR² metadata and records using the `mlcroissant` library.
- Examined available record sets and fields using their `@id` fields, ensuring reproducible references.
- Extracted and explored tabular data, filtering and normalizing numerical values.
- Visualized distributions and comparisons by group for deeper understanding.

This notebook can serve as a template for further, domain-specific analyses or as a reproducible benchmark for leveraging Croissant-encoded datasets.